# 🍽️ Day 03a: Classic Synchronization Problems

---

## 🎯 What You'll Master Today
- Producer-Consumer (Bounded Buffer)
- Readers-Writers Problem
- Dining Philosophers Problem
- Full Python implementations with semaphores and locks

---

In [ ]:
# ============================================
# 1. Producer-Consumer (Bounded Buffer)
# ============================================
import threading
import time
import random

class BoundedBuffer:
    def __init__(self, size: int):
        self.buffer = []
        self.size = size
        self.mutex = threading.Lock()            # protect buffer
        self.empty = threading.Semaphore(size)    # count of empty slots
        self.full = threading.Semaphore(0)        # count of full slots
    
    def produce(self, item, name):
        self.empty.acquire()    # wait for empty slot
        with self.mutex:        # exclusive access
            self.buffer.append(item)
            print(f"  📦 {name} produced {item} | Buffer: {self.buffer}")
        self.full.release()     # signal: one more full slot
    
    def consume(self, name):
        self.full.acquire()     # wait for full slot
        with self.mutex:
            item = self.buffer.pop(0)
            print(f"  🛒 {name} consumed {item} | Buffer: {self.buffer}")
        self.empty.release()    # signal: one more empty slot
        return item


buf = BoundedBuffer(size=3)

def producer(name, count):
    for i in range(count):
        buf.produce(f"item_{i}", name)
        time.sleep(random.uniform(0.05, 0.1))

def consumer(name, count):
    for _ in range(count):
        buf.consume(name)
        time.sleep(random.uniform(0.05, 0.15))

threads = [
    threading.Thread(target=producer, args=("P1", 4)),
    threading.Thread(target=consumer, args=("C1", 4)),
]
for t in threads: t.start()
for t in threads: t.join()
print("  Done!")

In [ ]:
# ============================================
# 2. Readers-Writers Problem (Writers Priority)
# ============================================
import threading
import time

class ReadWriteLock:
    def __init__(self):
        self._readers = 0
        self._readers_lock = threading.Lock()  # protect reader count
        self._write_lock = threading.Lock()     # exclusive for writers
    
    def acquire_read(self):
        with self._readers_lock:
            self._readers += 1
            if self._readers == 1:
                self._write_lock.acquire()  # first reader blocks writers
    
    def release_read(self):
        with self._readers_lock:
            self._readers -= 1
            if self._readers == 0:
                self._write_lock.release()  # last reader unblocks writers
    
    def acquire_write(self):
        self._write_lock.acquire()
    
    def release_write(self):
        self._write_lock.release()


shared_data = {"value": 0}
rw_lock = ReadWriteLock()

def reader(name):
    rw_lock.acquire_read()
    print(f"  📖 {name} reading: {shared_data['value']}")
    time.sleep(0.1)  # simulate read
    rw_lock.release_read()

def writer(name, value):
    rw_lock.acquire_write()
    shared_data["value"] = value
    print(f"  ✏️ {name} wrote: {value}")
    time.sleep(0.1)  # simulate write
    rw_lock.release_write()


threads = [
    threading.Thread(target=reader, args=("R1",)),
    threading.Thread(target=reader, args=("R2",)),
    threading.Thread(target=writer, args=("W1", 42)),
    threading.Thread(target=reader, args=("R3",)),
    threading.Thread(target=writer, args=("W2", 99)),
]
for t in threads: t.start()
for t in threads: t.join()
print(f"  Final value: {shared_data['value']}")

In [ ]:
# ============================================
# 3. Dining Philosophers
# ============================================
import threading
import time

NUM_PHILOSOPHERS = 5
forks = [threading.Lock() for _ in range(NUM_PHILOSOPHERS)]

def philosopher(pid: int, meals: int):
    # Solution: always pick lower-numbered fork first
    # This breaks the circular wait condition → no deadlock!
    left = pid
    right = (pid + 1) % NUM_PHILOSOPHERS
    first, second = (left, right) if left < right else (right, left)
    
    for meal in range(meals):
        # Think
        time.sleep(0.01)
        
        # Pick up forks (ordered to prevent deadlock)
        forks[first].acquire()
        forks[second].acquire()
        
        print(f"  🍝 Philosopher {pid} eating meal {meal + 1}")
        time.sleep(0.01)  # eat
        
        # Put down forks
        forks[second].release()
        forks[first].release()


threads = [threading.Thread(target=philosopher, args=(i, 2)) for i in range(NUM_PHILOSOPHERS)]
for t in threads: t.start()
for t in threads: t.join()
print("\n  All philosophers finished eating! No deadlock! ✅")

```
╔══════════════════════════════════════════════════════════════╗
║           DINING PHILOSOPHERS — DEADLOCK SOLUTIONS           ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. Resource ordering: always pick lower fork first         ║
║  2. Allow at most (N-1) to sit at table simultaneously     ║
║  3. Use a waiter (semaphore) to limit concurrent eaters     ║
║  4. Chandy/Misra: token-based solution                      ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Explain Producer-Consumer with semaphores | empty(N) + full(0) + mutex. Producer: wait(empty)→lock→produce→signal(full) |
| 2 | Readers-Writers starvation? | Basic solution starves writers (readers keep arriving). Fix: writer priority mode |
| 3 | Dining Philosophers deadlock? | All pick left fork → circular wait. Fix: resource ordering or limit concurrency |
| 4 | What is the circular wait condition? | Each process holds one resource and waits for one held by the next in a cycle |
| 5 | Bounded buffer vs unbounded? | Bounded needs both empty+full semaphores. Unbounded only needs full semaphore |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Producer-Consumer: empty + full sems + mutex          │
## │  • Readers-Writers: multiple readers OR one writer       │
## │  • Dining Philosophers: resource ordering = no deadlock  │
## │  • These patterns appear everywhere in real systems      │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Deadlocks** — conditions, detection, prevention, and Banker's algorithm